- verl 的主线设计是 HybridFlow:
    - 单进程 controller 表达 RL 算法控制流,
    - 多进程 worker 承担 actor、 rollout、critic、reward model、teacher model 等重计算。
    - OPD 没有另起一个完全独立的 trainer,而是接入已有 PPO 主干
- references
    - https://thinkingmachines.ai/blog/on-policy-distillation/
        - incorporate distillation loss as a reward，OPD-PG/reverse-KL reward
        - `use_policy_gradient=True`
            - `advantages=-distillation_losses.detach()`
        - student rollout 后, teacher 给 token-level logprob, 负的 KL estimator 作为 policy-gradient reward。
    - https://arxiv.org/abs/2306.13649
        - On-Policy Distillation of Language Models: Learning from Self-Generated Mistakes
        - ICLR 2024 的 GKD(Generalized Knowledge Distillation)/OPD，supervised/top-k fKL
        - 当 `use_policy_gradient=False` 时, distillation loss 直接作为 supervised loss 反传

### scripts

- https://github.com/verl-project/verl/blob/main/examples/on_policy_distillation_trainer/README.md
    - 文本 FSDP: examples/on_policy_distillation_trainer/run_qwen3_8b_fsdp.sh
        - 设置 distillation.enabled=True；
        - 默认 student Qwen3-8B，teacher Qwen3-32B；
        - 数据是 GSM8K + MATH，vLLM rollout，FSDP actor；
        - 默认 loss_mode=k1、use_policy_gradient=True、topk=64；
    - 文本 Megatron: examples/on_policy_distillation_trainer/run_qwen3_8b_megatron.sh
        - 同样是 Qwen3-8B 向 Qwen3-32B 蒸馏，但 actor 使用 Megatron。
        - 默认 loss_mode=forward_kl_topk、use_policy_gradient=False，更像直接 supervised KD。
    - VLM FSDP: examples/on_policy_distillation_trainer/run_qwen3_vl_8b_fsdp.sh
        - Qwen3-VL-8B-Instruct 向 Qwen3-VL-32B-Instruct 蒸馏，数据默认 Geo3K，并打开 data.image_key=images。
    - 多教师 MOPD: examples/on_policy_distillation_trainer/run_qwen3_8b_mopd_fsdp.sh
        - 一个 student 同时面对文本 teacher 和 VLM teacher，通过 distillation.teacher_key=data_source 路由样本。GSM8K 路由到文本 teacher，Geo3K 路由到 VLM teacher。

### config

- verl/trainer/config/ppo_trainer.yaml
    - distillation@distillation
        - verl/trainer/config/distillation/distillation.yaml
        - verl/workers/config/distillation.py
            - 配置类会校验 teacher GPU pool，并把 teacher inference 的 prompt 长度改成“student prompt + student response”，response 长度设成 1，这样 teacher 实际上是在完整学生轨迹上返回 prompt logprobs
    - 关键字段是 enabled、teacher_models、teacher_key、loss_mode、topk、use_task_rewards、use_policy_gradient。

### main_ppo.py

- distillation enabled
    - 创建 teacher_pool
- RayPPOTrainer.init_workers() 创建 MultiTeacherModelManager，把 teacher server client 传给 agent loop
- teacher manager 会按 teacher 的 world_size = num_replicas * per_replica_world_size 切分资源池并启动 vLLM/SGLang rollout replica
- 数据流（data flow）:
    - student rollout 生成 response，agent loop 在训练阶段调用 teacher，对 prompt_ids + response_ids 请求 prompt_logprobs。
    - vLLM 适配层把 prompt token ids 和 prompt logprobs 抽出来。
    - 最后这些teacher_ids、teacher_logprobs 被塞回 batch。

#### loss

OPD 的一轮训练可以写成：
$$
x \sim \mathcal{D}, \quad y \sim \pi_{\text{student}}(\cdot|x)
$$

对响应 token $a_t=y_t$，状态是

$$
s_t=(x,y_{<t}).
$$

teacher 不生成新答案，而是在学生已经生成的序列上计算：

$$
\log \pi_T(a_t|s_t)
$$

或 top-k 分布：

$$
{(i,\log\pi_T(i|s_t))}_{i\in \operatorname{TopK}_T(s_t)}.
$$

agent loop 把 prompt_ids + response_ids 发给 teacher（`verl/experimental/agent_loop/agent_loop.py`）。teacher manager 用 prompt_logprobs 请求teacher 对整条学生轨迹给 logprob。vLLM 侧抽出 prompt_ids 和 prompt_logprobs。这些最后进入 batch 的 teacher_ids、teacher_logprobs。

```python
self.teacher_server_manager.compute_teacher_logprobs_single(
                sequence_ids=prompt_ids + response_ids,           
```

- `use_policy_gradient=True`: 把 distillation loss 的负数当 advantage/reward，调用同一套 PPO clipped policy loss。代码里是`advantages=-distillation_losses.detach()`。这就是更“RL 化”的 OPD。
- `use_policy_gradient=False`: 直接对 distillation loss 反传，像在线版 KD/SFT。forward_kl_topk 就是这种典型模式，teacher 给 top-k 分布，student 对teacher top-k support 做 token-wise forward KL，
- 即：
  - k1/k2/k3/low_var_kl + use_policy_gradient=True: sample-based reverse KL OPD。
  - forward_kl_topk + use_policy_gradient=False: online KD / supervised distillation 风格的 forward KL。

> path 1: sample-estimator / reverse-KL 路径
- 拿当前 student 的 sampled-token logprob: $\log\pi_\theta(a_t|s_t)$，和 teacher 对同一个 token 的 logprob: $\log\pi_T(a_t|s_t)$
- 然后调用 kl_penalty。如果 loss_mode=k1，代码定义是：
$$
\hat D_t=\log\pi_\theta(a_t|s_t)-\log\pi_T(a_t|s_t).
$$
- 如果 use_policy_gradient=True，不直接反传这个标量，而是把它的负数作为 advantage: $A_t^{\text{distill}}=-\hat D_t.$
    - `advantages=-distillation_losses.detach()`
- 然后仍然进入 PPO clipped policy loss。也就是说 OPD-PG 的实际 actor loss 是：

$$
L_{\text{OPD-PG}}=\max(
-r_t A_t^{\text{distill}},
-\operatorname{clip}(r_t,1-\epsilon,1+\epsilon)A_t^{\text{distill}}
).
$$

> path 2: top-k forward-KL / supervised KD 路径

- teacher 给 top-k token id 和 logprob，student 在这些 id 上 gather 自己的 logprob:

$$
\log\pi_\theta(i|s_t), \quad i\in \operatorname{TopK}_T(s_t).
$$

```python
student_log_probs = F.log_softmax(student_logits, dim=-1)
student_topk_log_probs = torch.gather(student_log_probs, dim=-1, index=teacher_topk_ids)
distillation_losses = kl_divergence(log_q=student_topk_log_probs, log_p=teacher_topk_log_probs)
```

学上是稀疏 forward KL:

$$
D_{\mathrm{KL}}(\pi_T||\pi_\theta)
\approx
\sum_{i\in \operatorname{TopK}T}
\pi_T(i|s_t)
\left[
\log\pi_T(i|s_t)-\log\pi\theta(i|s_t)
\right].
$$

#### PPO、GRPO、OPD

PPO、GRPO、OPD 共享同一条 trainer 骨架: rollout，重算 old_log_probs，可选 ref/critic/reward，算 advantage，最后 update_actor。标准 PPO loss 是 ratio clipping:

$$
r_t = \exp(\log \pi_\theta(a_t) - \log \pi_{\text{old}}(a_t))
$$

$$
L = \max(-r_t A_t,\ -\operatorname{clip}(r_t, 1-\epsilon, 1+\epsilon)A_t)
$$

```python
# core_algos.py
pg_losses1 = -advantages * ratio
pg_losses2 = -advantages * torch.clamp(ratio, 1 - cliprange_low, 1 + cliprange_high)
clip_pg_losses1 = torch.maximum(pg_losses1, pg_losses2)
```

直觉是：如果 $A_t>0$，这个 token 比较好，模型会想增大它的概率，但 $r_t$ 超过 $1+\epsilon$ 后就不再给额外收益；如果 $A_t<0$，这个 token 比较差，模型会想降低它的概率，但 $r_t$ 低于 $1-\epsilon$ 后也不再允许继续从 loss 里获利。max 选的是“更保守、更大的 loss”，防止单步更新把策略推太远。

-----
$$
J(\theta)=\mathbb{E}_{x \sim \mathcal{D},\ y \sim \pi_{\theta_{\text{old}}}(\cdot|x),\ t \in [1, |y|]}\left[\min(r_t A_t,\ \operatorname{clip}(r_t,1-\epsilon,1+\epsilon)A_t)\right]
$$

在目标函数中通过 $\min$ 实现悲观限界（Pessimistic Bound），

$$
L(\theta)=-J(\theta)
=\max(-r_t A_t,\ -\operatorname{clip}(r_t,1-\epsilon,1+\epsilon)A_t).
$$

$-\min(x,y)=\max(-x,-y)$

-----
区别在于优化信号

- PPO: adv_estimator=gae 时需要 critic value，advantage 来自 reward + value bootstrap；
- GRPO: 不需要 critic，通常每个 prompt 采多条 response，在同组内按 outcome reward 做均值/方差归一化，再广播到 token；
    - PPO surrogate + group-relative advantage
- OPD: reward 不是外部标量为主，而是 teacher 给的 token 级分布差异。若 use_task_rewards=False，普通 PPO/GRPO 的 task reward policy loss 会被置零，只保留 distillation term；
    - PPO trainer 基础设施 + teacher KL/logprob 监督

### reuse

$$
\log\frac{\pi_T(a_t|s_t)}{\pi_\theta(a_t|s_t)}=\log\pi_T(a_t|s_t)-\log\pi_\theta(a_t|s_t).
$$

这在代码里作为 reward/advantage 使用。对应的 loss 记成相反数：

$$
d_t=\log\pi_\theta(a_t|s_t)-\log\pi_T(a_t|s_t).
$$

如果 student 给某 token 很高概率，但 teacher 给低概率，那么：

$$
\log\pi_\theta-\log\pi_T > 0
$$

loss 高，应该压低它。如果 teacher 比 student 更认可这个 sampled token，那么：

$$
\log\pi_\theta-\log\pi_T < 0
$$

它变成正向学习信号。

```
return logprob - ref_logprob
```

> 最小化 reverse KL 等价于一种 RL 最大化问题（reverse KL 的梯度就是 policy gradient）

$$
\begin{split}
\nabla_{\theta}\mathcal{L}_{RL} &= - \mathbb{E}_{y \sim \pi_{\theta}} \left[ \nabla_{\theta}\log\pi_{\theta}(y) \cdot A(y) \right]\\
\mathcal{L}_{OPD}(\theta) &= D_{KL}(\pi_{\theta} || \pi_{E}) = \sum_{y} \pi_{\theta}(y) \log \frac{\pi_{\theta}(y)}{\pi_{E}(y)}
\end{split}
$$
- $\nabla_{\theta} D_{KL}(\pi_{\theta} || \pi_{E}) = \sum_{y} \left( \nabla_{\theta}\pi_{\theta}(y) \cdot \log \frac{\pi_{\theta}(y)}{\pi_{E}(y)} + \pi_{\theta}(y) \cdot \nabla_{\theta} \left[ \log \pi_{\theta}(y) - \log \pi_{E}(y) \right] \right)$
    - 右侧部分 $\pi_{\theta}(y) \cdot \nabla_{\theta}\log\pi_{\theta}(y) \rightarrow \sum_{y} \nabla_{\theta}\pi_{\theta}(y) = \nabla_{\theta} \left( \sum_{y} \pi_{\theta}(y) \right)=0$
- $\nabla_{\theta} D_{KL} = \sum_{y} \nabla_{\theta}\pi_{\theta}(y) \cdot \log \frac{\pi_{\theta}(y)}{\pi_{E}(y)}=\sum_{y} \pi_{\theta}(y) \nabla_{\theta}\log\pi_{\theta}(y) \cdot \log \frac{\pi_{\theta}(y)}{\pi_{E}(y)}= - \mathbb{E}_{y \sim \pi_{\theta}} \left[ \nabla_{\theta}\log\pi_{\theta}(y) \cdot \log \frac{\pi_{E}(y)}{\pi_{\theta}(y)} \right]$
- 定义：$A(y) = \log \frac{\pi_{E}(y)}{\pi_{\theta}(y)}$
    - $A_t = \text{sg}\left[\log\frac{\pi_{E_i}(y_t|x, y_{<t})}{\pi_\theta(y_t|x, y_{\le t})}\right]$

### rkl vs. fkl

> “对谁取期望”

- 设 teacher 分布是 $p(a|s)=\pi_T(a|s)$，student 分布是 $q(a|s)=\pi_\theta(a|s)$。
- Forward KL 是：
$$
D_{\mathrm{KL}}(p||q)=\sum_a p(a|s)\log\frac{p(a|s)}{q(a|s)}=\mathbb{E}_{a\sim p}[\log p(a|s)-\log q(a|s)].
$$
- Reverse KL 是：
$$
D_{\mathrm{KL}}(q||p)=\sum_a q(a|s)\log\frac{q(a|s)}{p(a|s)}=\mathbb{E}_{a\sim q}[\log q(a|s)-\log p(a|s)].
$$
- 即谁在求和权重里（求和项的权重），谁就是 KL 的第一个分布。
    - 如果样本 $a_t$ 来自 student，也就是$a_t \sim q(\cdot|s_t),$
    - 然后每个样本上计算 $\log q(a_t|s_t)-\log p(a_t|s_t),$
    - 那么大量样本平均后就是：
  $$
  \frac{1}{N}\sum_{i=1}^N[\log q(a_i|s)-\log p(a_i|s)]\approx
  \mathbb{E}_{a\sim q}[\log q(a|s)-\log p(a|s)]=D_{\mathrm{KL}}(q||p).
  $$
- student-to-teacher reverse KL。
    - Monte Carlo 估计里，“样本来自谁”决定了你在估计哪个期望。例如：$\mathbb{E}_{a\sim q}[f(a)]$
    - 要用 $a\sim q$ 的样本估计。OPD 里 student rollout 产生 token，所以采样测度是 $q=\pi_\theta$。teacher 只是给这个 student token 打分，提供 $\log p(a_t|s_t)$。显然不是问“teacher 会生成哪些 token”，而是在问“student 已经生成的 token，在 teacher 看来概率高不高”。
- 所以 OPD-rKL 的每个 token loss 可以理解成：

$$
\hat D_t=\log\pi_\theta(a_t|s_t)-\log\pi_T(a_t|s_t),
\quad a_t\sim \pi_\theta(\cdot|s_t).
$$

- 取期望后：

    $$
    \mathbb{E}_{a_t\sim\pi_\theta}[\hat D_t]=D_{\mathrm{KL}}(\pi_\theta(\cdot|s_t)||\pi_T(\cdot|s_t)).
    $$
    
    - 不是“只要样本来自 student 就一定是 reverse KL”，而是“样本来自 student，并且你估计的是 $\log\pi_\theta-\log\pi_T$ 这个 log-ratio（log-density ratio，这个 student token 在 teacher 看来相对合理吗？），所以它是 reverse KL”。

- 如果要做 forward KL，就需要 teacher 分布作为权重：

$$
D_{\mathrm{KL}}(\pi_T||\pi_\theta)=  \mathbb{E}{a\sim\pi_T}[\log\pi_T(a|s)-\log\pi\theta(a|s)].
$$

- 这通常要求 teacher 采样 token，或者 teacher 给出一批 token 的概率分布。verl 的 forward_kl_topk 就是后一种：teacher 给 top-k token 及其概率，student 在这些teacher top-k token 上 gather 自己的 logprob，然后显式求：

$$
\sum_{a\in \operatorname{TopK}T}\pi_T(a|s)[\log\pi_T(a|s)-\log\pi\theta(a|s)].
$$

  所以差别是：

  OPD-rKL: student 采样，teacher 评价 student token。

  $$
  a\sim\pi_\theta,\quad
  \log\pi_\theta(a)-\log\pi_T(a)
  \Rightarrow
  D_{\mathrm{KL}}(\pi_\theta||\pi_T)
  $$

  Online KD / top-k KD: teacher 给目标分布，student 拟合 teacher 分布。

  $$
  a \text{ weighted by } \pi_T(a),\quad
  \log\pi_T(a)-\log\pi_\theta(a)
  \Rightarrow
  D_{\mathrm{KL}}(\pi_T||\pi_\theta)
  $$

- 为什么第一种是 reverse KL（student-to-teacher）？因为样本 $a_t$ 来自 student，而不是 teacher。对 k1 来说：

$$
\hat D_t=\log\pi_\theta(a_t|s_t)-\log\pi_T(a_t|s_t),
\quad a_t\sim\pi_\theta(\cdot|s_t).
$$

取期望：

$$
\begin{aligned}
\mathbb{E}_{a \sim \pi_\theta(\cdot \mid s)}
\left[\log \pi_\theta(a \mid s)-\log \pi_T(a \mid s)\right]
&=\sum_a \pi_\theta(a \mid s)\log \frac{\pi_\theta(a \mid s)}{\pi_T(a \mid s)}
\\
&=D_{\mathrm{KL}}\!\left(\pi_\theta(\cdot \mid s)\,\|\,\pi_T(\cdot \mid s)\right).
\end{aligned}
$$

- 和 SFT / 在线 KD 的区别是，SFT 用固定数据 token $a\sim p_{\text{data}}$，优化 $L_{\text{SFT}}=-\log\pi_\theta(a|s).$。这等价于最小化 $D_{\mathrm{KL}}(p_{\text{data}}||\pi_\theta)$。差一个与模型无关的常数。它是 data/teacher 到 student 的方向，偏 forward KL。
- 在线 KD top-k 虽然状态 $s_t$ 来自 student rollout，但动作分布目标来自 teacher top-k，优化的是：
    - $D_{\mathrm{KL}}(\pi_T(\cdot|s_t)||\pi_\theta(\cdot|s_t)).$
    - 所以它“在线”在状态分布上，但 KL 方向仍是 teacher-to-student。
- OPD-rKL 则是 student 自己采样 token，teacher 只评价这些 token: $D_{\mathrm{KL}}(\pi_\theta(\cdot|s_t)||\pi_T(\cdot|s_t)).$
- 这个方向更像“惩罚 student 把概率质量放在 teacher 不认可的地方”，而不是“要求 student 覆盖 teacher 的所有高概率模式”。这也是为什么代码里 k1 不能在 use_policy_gradient=False 下直接反传: 直接对 $\log\pi_\theta-\log\pi_T$ 求梯度会变成 $\nabla\log\pi_\theta$，teacher 项不参与梯度，语义是错的；配置类对此直接报错，verl/workers/config/distillation.py。

```python
if not self.use_policy_gradient and self.loss_mode == "k1":
    raise ValueError(
        "Directly backpropagating k1 loss is incorrect since gradient of k1 loss"
        " wrt model weights does not depend on teacher log probabilities."
    )
```